In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.optimize import nnls
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv("data/processed/mmm_clean.csv", parse_dates=["Date"])
channels = ["TikTok", "Facebook", "Google Ads"]

In [2]:
#efficiency trend

yearly = df.groupby("year").agg(spend=("total_spend", "sum"), sales=("Sales", "sum"))
yearly["revenue_per_dollar"] = yearly.sales / yearly.spend
yearly.round(2)

,spend,sales,revenue_per_dollar
year,,,
2018,384561.83,569560.78,1.48
2019,272181.71,505560.27,1.86
2020,372067.48,572372.83,1.54
2021,307292.03,486134.42,1.58


In [3]:
#channel mix shift

mix = df.groupby("year")[channels].sum()
mix_pct = mix.div(mix.sum(axis=1), axis=0) * 100
mix_pct.round(1)

,TikTok,Facebook,Google Ads
year,,,
2018,48.8,32.9,18.3
2019,35.1,34.0,30.8
2020,47.0,34.0,19.0
2021,42.7,31.6,25.7


In [4]:
def adstock(x, decay=0.5):
    y = x.copy()
    for i in range(1, len(y)):
        y[i] += decay * y[i - 1]
    return y

X = pd.DataFrame({c: adstock(df[c].values) for c in channels})
print("Correlation between adstocked channels:\n", X.corr().round(2))

vif = pd.DataFrame({"channel": channels,
                     "VIF": [variance_inflation_factor(X.values, i) for i in range(len(channels))]})
print("\nVariance Inflation Factor (rule of thumb: above 5 means a channel's spend can't be "
      "cleanly separated from the others' -- that's a structural reason for specs to disagree "
      "on that channel, not a modeling mistake):\n", vif.round(1))

# Spec A: linear, unconstrained (OLS)
spec_a = sm.OLS(df["Sales"], sm.add_constant(X)).fit()

# Spec B: linear, non-negative (NNLS) -- rules out a channel "hurting" sales by construction
spec_b_coefs, _ = nnls(X.values, df["Sales"].values)
pred_b = X.values @ spec_b_coefs
r2_b = 1 - ((df["Sales"].values - pred_b) ** 2).sum() / ((df["Sales"].values - df["Sales"].values.mean()) ** 2).sum()

# Spec C: same adstocked spend, but with a saturation (diminishing-returns) curve on top --
# the other standard MMM transform besides adstock. log1p is the simplest defensible saturation
# curve: it says the 1000th dollar of spend buys less incremental sales than the 100th did.
X_sat = np.log1p(X)
spec_c = sm.OLS(df["Sales"], sm.add_constant(X_sat)).fit()

print(f"\nSpec A (OLS, linear)     R2={spec_a.rsquared:.3f}\n", spec_a.params.round(3))
print(f"\nSpec B (NNLS, linear)    R2={r2_b:.3f}\n", dict(zip(channels, spec_b_coefs.round(3))))
print(f"\nSpec C (OLS, saturated)  R2={spec_c.rsquared:.3f}\n", spec_c.params.round(3))

Correlation between adstocked channels:
             TikTok  Facebook  Google Ads
TikTok        1.00      0.06        0.02
Facebook      0.06      1.00       -0.11
Google Ads    0.02     -0.11        1.00

Variance Inflation Factor (rule of thumb: above 5 means a channel's spend can't be cleanly separated from the others' -- that's a structural reason for specs to disagree on that channel, not a modeling mistake):
       channel  VIF
0      TikTok  2.0
1    Facebook  2.7
2  Google Ads  3.0

Spec A (OLS, linear)     R2=0.898
 const         3876.401
TikTok           0.362
Facebook         0.434
Google Ads       0.908
dtype: float64

Spec B (NNLS, linear)    R2=0.746
 {'TikTok': np.float64(0.408), 'Facebook': np.float64(0.621), 'Google Ads': np.float64(1.733)}

Spec C (OLS, saturated)  R2=0.488
 const        -5516.410
TikTok         813.363
Facebook       456.199
Google Ads     793.307
dtype: float64


In [5]:
rng = np.random.default_rng(0)
n_boot = 500
boot_a = {c: [] for c in channels}
boot_b = {c: [] for c in channels}

for _ in range(n_boot):
    sample = rng.integers(0, len(df), len(df))
    Xs, ys = X.values[sample], df["Sales"].values[sample]
    a_fit = sm.OLS(ys, sm.add_constant(Xs)).fit()
    b_fit, _ = nnls(Xs, ys)
    for i, c in enumerate(channels):
        boot_a[c].append(a_fit.params[i + 1])
        boot_b[c].append(b_fit[i])

ci_rows = []
for c in channels:
    a_lo, a_hi = np.percentile(boot_a[c], [5, 95])
    b_lo, b_hi = np.percentile(boot_b[c], [5, 95])
    ci_rows.append({"channel": c, "spec_a_low": a_lo, "spec_a_high": a_hi,
                     "spec_b_low": b_lo, "spec_b_high": b_hi})
    print(f"{c}: Spec A 90% CI [{a_lo:.2f}, {a_hi:.2f}]  |  Spec B 90% CI [{b_lo:.2f}, {b_hi:.2f}]")

ci_df = pd.DataFrame(ci_rows)

TikTok: Spec A 90% CI [0.35, 0.38]  |  Spec B 90% CI [0.38, 0.44]
Facebook: Spec A 90% CI [0.40, 0.47]  |  Spec B 90% CI [0.57, 0.67]
Google Ads: Spec A 90% CI [0.81, 1.01]  |  Spec B 90% CI [1.65, 1.81]


In [6]:
decomp = pd.DataFrame({"year": df["year"], "Baseline": spec_a.params["const"]})
for c in channels:
    decomp[c] = spec_a.params[c] * X[c].values

decomp_yearly = decomp.groupby("year").sum().reset_index()
decomp_long = decomp_yearly.melt(id_vars="year", var_name="component", value_name="attributed_sales")
decomp_long.to_csv("data/processed/sales_decomposition.csv", index=False)
decomp_long

,year,component,attributed_sales
0,2018,Baseline,201572.832164
1,2019,Baseline,201572.832164
2,2020,Baseline,201572.832164
3,2021,Baseline,170561.627215
4,2018,TikTok,134244.958423
5,2019,TikTok,65722.400783
6,2020,TikTok,131596.655257
7,2021,TikTok,94347.833183
8,2018,Facebook,105326.277147
9,2019,Facebook,84050.405723


In [8]:
yearly.reset_index().to_csv("data/processed/efficiency_trend.csv", index=False)

mix_long = mix_pct.reset_index().melt(id_vars="year", var_name="channel", value_name="pct_of_spend")
mix_long.to_csv("data/processed/mix_shift.csv", index=False)

comparison = pd.DataFrame({"channel": channels,
                            "Spec A (OLS)": spec_a.params[channels].values,
                            "Spec B (NNLS)": spec_b_coefs})
comp_long = comparison.melt(id_vars="channel", var_name="spec", value_name="coefficient")
comp_long = comp_long.merge(ci_df, on="channel")
comp_long["ci_low"] = np.where(comp_long["spec"] == "Spec A (OLS)", comp_long["spec_a_low"], comp_long["spec_b_low"])
comp_long["ci_high"] = np.where(comp_long["spec"] == "Spec A (OLS)", comp_long["spec_a_high"], comp_long["spec_b_high"])
comp_long[["channel", "spec", "coefficient", "ci_low", "ci_high"]].to_csv(
    "data/processed/attribution_comparison.csv", index=False)